# Sheet 3.1: RL basics: returns, policy gradients, DQN

**Roadmap topic 3.1** (Fanout 07.01-07.03). Read first: Spinning Up *Key Concepts* and *Intro to Policy Optimization*, HF Deep RL course units 1-4, Sutton & Barto ch. 3, 6, 13, Karpathy's *Pong from Pixels*.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells. Everything runs on CPU (`pip install gymnasium` if missing).

In [ ]:
import math, random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import gymnasium as gym
torch.manual_seed(0); np.random.seed(0)

## Exercise 1: Discounted returns across episode boundaries

Rollouts from vectorised environments are flat arrays where episodes end at `dones[t] = True` (the reward at `t` is the last reward of that episode). Implement `discounted_returns(rewards, dones, gamma, last_value=0.0)`:
`G_t = r_t + γ·G_{t+1}·(1 - done_t)`, and the final `G_{T}` bootstraps from `last_value` if the last step is not terminal. One backward pass, O(T).

In [ ]:
def discounted_returns(rewards, dones, gamma, last_value=0.0):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    r = np.array([1., 0., 2., 1., 1., 3.]); d = np.array([0, 0, 1, 0, 1, 0], dtype=float)
    G = discounted_returns(r, d, 0.5, last_value=4.0)
    assert np.allclose(G, [1 + 0.5 * (0 + 0.5 * 2), 0 + 0.5 * 2, 2, 1 + 0.5 * 1, 1, 3 + 0.5 * 4]), G
    assert np.allclose(discounted_returns(np.ones(4), np.zeros(4), 1.0), [4, 3, 2, 1])
    print("PASS")
_check1()

## Exercise 2: The policy-gradient estimator, its bias and its variance

A one-step problem: a 4-armed bandit with softmax policy `π_θ = softmax(θ)` and noisy rewards `r ~ N(μ_a, 1)` with `μ = [1, 2, 3, 10]` (note the large offset of arm 3).
- `exact_grad(theta)`: the true `∇_θ E[r]` (derive it).
- `pg_samples(theta, n, baseline)`: `n` single-sample REINFORCE estimates `(r - b)·∇_θ log π(a)` as an `(n, 4)` array, where `b = baseline` (a constant).

The check shows both estimators are unbiased but the baseline cuts variance.

In [ ]:
MU = np.array([1., 2., 3., 10.])
def exact_grad(theta):
    # YOUR CODE
    raise NotImplementedError
def pg_samples(theta, n, baseline=0.0, rng=np.random.default_rng(0)):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    th = np.array([0.5, 0.0, -0.3, -1.0]); ex = exact_grad(th)
    tt = torch.tensor(th, requires_grad=True); (torch.softmax(tt, 0) @ torch.tensor(MU)).backward()
    assert np.allclose(ex, tt.grad.numpy()), "exact gradient wrong"
    n = 200_000
    g0 = pg_samples(th, n, 0.0, np.random.default_rng(1)); gb = pg_samples(th, n, float(_b := (np.exp(th) / np.exp(th).sum()) @ MU), np.random.default_rng(2))
    for g in (g0, gb):
        assert np.abs(g.mean(0) - ex).max() < 4 * g.std(0).max() / math.sqrt(n), "estimator looks biased"
    v0, vb = g0.var(0).sum(), gb.var(0).sum()
    assert vb < v0 / 2, f"baseline should cut variance a lot: {v0:.2f} -> {vb:.2f}"
    print(f"PASS  total variance without baseline {v0:.2f}, with baseline {vb:.2f}")
_check2()

**Written (2a):** Prove `E[b·∇ log π(a)] = 0` for any constant `b`. Why does a *state-dependent* baseline `b(s)` keep this property but an *action-dependent* one doesn't?

**Your answer:**

## Exercise 3: REINFORCE with a learned baseline on CartPole

`reinforce(env_id, episodes, baseline, seed)` trains a small policy MLP (and, if `baseline=True`, a value MLP regressed on returns, used as `b(s)`) with Adam. Normalise nothing else. Return the list of episode returns.

In [ ]:
def reinforce(env_id="CartPole-v1", episodes=600, baseline=True, gamma=0.99, lr=1e-2, seed=0):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    r = reinforce(baseline=True, seed=0)
    best = max(np.mean(r[i:i + 20]) for i in range(len(r) - 20))
    assert best >= 400, f"best 20-episode average {best:.0f} < 400; check your gradient sign, returns and baseline"
    print(f"PASS  best 20-episode average return {best:.0f}")
_check3()

Then compare `baseline=False` vs `True` over 5 seeds each and plot mean ± std of returns.

## Exercise 4: DQN pieces you must get exactly right

- `ReplayBuffer(capacity, obs_dim)` with `add(s, a, r, s2, done)` (ring buffer) and `sample(batch, rng)` returning tensors.
- `td_target(r, s2, done, q_target, gamma, double=False, q_online=None)`: `r + γ(1-done)·max_a Q_target(s2, a)`; with `double=True` choose the action with `q_online` and evaluate with `q_target` (Double DQN).
- `polyak_update(target, online, tau)`: `θ_t ← (1-τ)θ_t + τθ`.

**Important:** in Gymnasium, `done` for bootstrapping must be `terminated`, **not** `terminated or truncated`. Explain why in the written answer.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity, obs_dim): raise NotImplementedError
    def add(self, s, a, r, s2, done): raise NotImplementedError
    def sample(self, batch, rng): raise NotImplementedError
    def __len__(self): raise NotImplementedError
def td_target(r, s2, done, q_target, gamma, double=False, q_online=None): raise NotImplementedError
def polyak_update(target, online, tau): raise NotImplementedError

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    b = ReplayBuffer(5, 2)
    for i in range(8): b.add([i, i], i % 2, float(i), [i + 1, i + 1], float(i == 7))
    assert len(b) == 5 and sorted(b.S[:, 0].tolist()) == [3, 4, 5, 6, 7], "ring buffer must keep the newest 5"
    S, A, R, S2, D = b.sample(64, np.random.default_rng(0))
    assert S.shape == (64, 2) and A.dtype == torch.int64 and torch.all(S2[:, 0] == S[:, 0] + 1)
    qt = lambda s: torch.stack([s[:, 0], 2 * s[:, 0], -s[:, 0]], 1); qo = lambda s: torch.stack([s[:, 0], -s[:, 0], 3 * s[:, 0]], 1)
    s2 = torch.tensor([[1.], [2.]]); r = torch.tensor([1., 1.]); d = torch.tensor([0., 1.])
    assert torch.allclose(td_target(r, s2, d, qt, 0.9), torch.tensor([1 + 0.9 * 2, 1.]))
    assert torch.allclose(td_target(r, s2, d, qt, 0.9, double=True, q_online=qo), torch.tensor([1 + 0.9 * -1, 1.]))
    t, o = nn.Linear(2, 2), nn.Linear(2, 2); t0 = t.weight.detach().clone(); polyak_update(t, o, 0.1)
    assert torch.allclose(t.weight, 0.9 * t0 + 0.1 * o.weight)
    print("PASS")
_check4()

**Written (4a):** CartPole-v1 truncates at 500 steps. What goes wrong in the Q-values if you treat truncation as a terminal state?

### Train it (CPU, ~10 min)
Write the DQN loop (ε-greedy from 1.0 → 0.05 over 10k steps, buffer 50k, batch 128, Adam 5e-4, Huber loss, target update τ = 0.005) and reach a 100-episode average ≥ 475 on CartPole-v1. Then LunarLander-v3 (`pip install "gymnasium[box2d]"`) to an average ≥ 200. Ablate: no target network (τ = 1), and `done = terminated or truncated`.

## Exercise 5: Reporting RL results honestly

- `iqm(x)`: interquartile mean (mean of the middle 50%, as in `scipy.stats.trim_mean(x, 0.25)`).
- `bootstrap_ci(scores, stat, n_boot, alpha, seed)`: percentile CI for `stat` over **runs** (resample runs with replacement).

In [ ]:
def iqm(x): raise NotImplementedError
def bootstrap_ci(scores, stat, n_boot=5000, alpha=0.05, seed=0): raise NotImplementedError

In [ ]:
# ---- CHECK 5 (do not edit) ----
def _check5():
    from scipy.stats import trim_mean
    x = np.random.default_rng(0).normal(size=40) ** 3
    assert abs(iqm(x) - trim_mean(x, 0.25)) < 1e-12
    runs = np.array([480, 500, 120, 495, 470, 500, 30, 488, 499, 476.])
    lo, hi = bootstrap_ci(runs, iqm)
    assert lo <= iqm(runs) <= hi, (lo, hi)
    mlo, mhi = bootstrap_ci(runs, np.mean)
    assert iqm(runs) > runs.mean() + 50, "IQM should be far less affected by the two failed runs than the mean"
    print(f"PASS  IQM {iqm(runs):.0f} [{lo:.0f}, {hi:.0f}]  mean {runs.mean():.0f} [{mlo:.0f}, {mhi:.0f}]")
_check5()

## Checkpoint
Derive the policy gradient theorem with the log-derivative trick on paper, and explain why a baseline doesn't bias it. Then explain what the target network in DQN is protecting against.

---
### Hints for self-grading
- 2a: `E[∇log π] = Σ π ∇log π = ∇Σ π = ∇1 = 0`. With `b(s)` the sum is still over actions for a fixed `s`. With `b(s,a)` you can't pull `b` out of the sum.
- 4a: the agent learns that states near step 500 have value ~0, which is false (the pole is still up). Bootstrapping must continue through time-limit truncation.